In [0]:
USE CATALOG <yourcatalog>;
USE SCHEMA <yourschema;

## create silver_nurse_staffing

In [0]:
CREATE OR REPLACE TABLE silver_nurse_staffing AS
SELECT
  CAST(month AS DATE) AS month,
  UPPER(TRIM(hospital_id)) AS hospital_id,
  UPPER(TRIM(department_id)) AS department_id,
  CAST(budgeted_nurse_fte AS INT) AS budgeted_nurse_fte,
  CAST(filled_nurse_fte AS INT) AS filled_nurse_fte,
  CAST(open_nurse_positions AS INT) AS open_nurse_positions,
  CAST(nurse_overtime_hours AS DECIMAL(10,1)) AS nurse_overtime_hours,
  CAST(agency_nurse_hours AS DECIMAL(10,1)) AS agency_nurse_hours,
  INITCAP(TRIM(staffing_status)) AS staffing_status
FROM bronze_nurse_staffing
WHERE hospital_id IS NOT NULL
  AND department_id IS NOT NULL
  AND month IS NOT NULL
  AND budgeted_nurse_fte >= 0
  AND filled_nurse_fte >= 0
  AND filled_nurse_fte <= budgeted_nurse_fte;

num_affected_rows,num_inserted_rows


## create, clean monlthy_labor_cost

In [0]:
CREATE OR REPLACE TABLE silver_monthly_labor_cost AS
SELECT
  CAST(month AS DATE) AS month,
  UPPER(TRIM(hospital_id)) AS hospital_id,
  UPPER(TRIM(department_id)) AS department_id,
  CAST(regular_pay AS DECIMAL(12,2)) AS regular_pay,
  CAST(overtime_pay AS DECIMAL(12,2)) AS overtime_pay,
  CAST(agency_pay AS DECIMAL(12,2)) AS agency_pay,
  CAST(benefits_cost AS DECIMAL(12,2)) AS benefits_cost,
  CAST(paid_hours AS DECIMAL(12,1)) AS paid_hours
FROM monthly_labor_cost
WHERE
  hospital_id IS NOT NULL
  AND department_id IS NOT NULL
  AND month IS NOT NULL
  AND regular_pay >= 0
  AND overtime_pay >= 0
  AND benefits_cost >= 0
  AND paid_hours >= 0;

num_affected_rows,num_inserted_rows


## create, clean silver monlthy patient volume

In [0]:
CREATE OR REPLACE TABLE silver_monthly_patient_volume AS
WITH ranked_patient_volume AS (
  SELECT
    CAST(month AS DATE) AS month,
    UPPER(TRIM(hospital_id)) AS hospital_id,
    UPPER(TRIM(department_id)) AS department_id,
    CAST(patient_encounters AS INT) AS patient_encounters,
    CAST(patient_days AS INT) AS patient_days,
    CAST(procedures AS INT) AS procedures,
    CAST(avg_wait_minutes AS DECIMAL(10,1)) AS avg_wait_minutes,

    ROW_NUMBER() OVER (
      PARTITION BY month, hospital_id, department_id
      ORDER BY patient_encounters DESC, patient_days DESC, procedures DESC
    ) AS rn

  FROM monthly_patient_volume
)
SELECT
  month,
  hospital_id,
  department_id,
  patient_encounters,
  patient_days,
  procedures,
  avg_wait_minutes
FROM ranked_patient_volume
WHERE rn = 1;

num_affected_rows,num_inserted_rows


validate all tables

In [0]:
SELECT 'silver_nurse_staffing' AS table_name, COUNT(*) AS row_count
FROM silver_nurse_staffing
UNION ALL
SELECT 'silver_monthly_labor_cost', COUNT(*)
FROM silver_monthly_labor_cost
UNION ALL
SELECT 'silver_monthly_patient_volume', COUNT(*)
FROM silver_monthly_patient_volume;

table_name,row_count
silver_nurse_staffing,288
silver_monthly_labor_cost,288
silver_monthly_patient_volume,288


Validate Patient-Volume Duplicates

In [0]:
SELECT
  month,
  hospital_id,
  department_id,
  COUNT(*) AS duplicate_count
FROM silver_monthly_patient_volume
GROUP BY month, hospital_id, department_id
HAVING COUNT(*) > 1;

month,hospital_id,department_id,duplicate_count


Validate Staffing Gap Logic

In [0]:
SELECT *
FROM silver_nurse_staffing
WHERE budgeted_nurse_fte - filled_nurse_fte <> open_nurse_positions;

month,hospital_id,department_id,budgeted_nurse_fte,filled_nurse_fte,open_nurse_positions,nurse_overtime_hours,agency_nurse_hours,staffing_status


Validate Join Keys for the Gold Layer

In [0]:
SELECT
  s.month,
  s.hospital_id,
  s.department_id
FROM silver_nurse_staffing AS s
LEFT ANTI JOIN silver_monthly_labor_cost AS l
  ON s.month = l.month
  AND s.hospital_id = l.hospital_id
  AND s.department_id = l.department_id;

month,hospital_id,department_id


## create gold staffing_master

In [0]:
CREATE OR REPLACE TABLE workspace.healthcare_staffing.gold_staffing_tableau_master AS
SELECT
  s.month,
  s.hospital_id,
  s.department_id,

  -- Staffing measures
  s.budgeted_nurse_fte,
  s.filled_nurse_fte,
  s.open_nurse_positions,
  s.budgeted_nurse_fte - s.filled_nurse_fte AS nurse_staffing_gap,

  ROUND(
    (s.budgeted_nurse_fte - s.filled_nurse_fte)
      / NULLIF(s.budgeted_nurse_fte, 0),
    4
  ) AS vacancy_rate,

  s.nurse_overtime_hours,
  s.agency_nurse_hours,
  s.staffing_status,

  -- Labor-cost measures
  l.regular_pay,
  l.overtime_pay,
  l.agency_pay,
  l.benefits_cost,
  l.paid_hours,

  l.regular_pay
    + l.overtime_pay
    + l.agency_pay
    + l.benefits_cost AS total_labor_cost,

  ROUND(
    l.overtime_pay /
      NULLIF(
        l.regular_pay + l.overtime_pay + l.agency_pay + l.benefits_cost,
        0
      ),
    4
  ) AS overtime_pay_share,

  ROUND(
    l.agency_pay /
      NULLIF(
        l.regular_pay + l.overtime_pay + l.agency_pay + l.benefits_cost,
        0
      ),
    4
  ) AS agency_pay_share,

  -- Patient-volume and workload measures
  p.patient_encounters,
  p.patient_days,
  p.procedures,
  p.avg_wait_minutes,

  ROUND(
    p.patient_encounters / NULLIF(l.paid_hours, 0),
    2
  ) AS encounters_per_paid_hour,

  ROUND(
    (l.regular_pay + l.overtime_pay + l.agency_pay + l.benefits_cost)
      / NULLIF(p.patient_encounters, 0),
    2
  ) AS labor_cost_per_encounter,

  ROUND(
    s.nurse_overtime_hours / NULLIF(s.filled_nurse_fte, 0),
    2
  ) AS overtime_hours_per_filled_fte,

  ROUND(
    s.agency_nurse_hours / NULLIF(s.filled_nurse_fte, 0),
    2
  ) AS agency_hours_per_filled_fte

FROM workspace.healthcare_staffing.silver_nurse_staffing AS s

INNER JOIN workspace.healthcare_staffing.silver_monthly_labor_cost AS l
  ON s.month = l.month
  AND s.hospital_id = l.hospital_id
  AND s.department_id = l.department_id

INNER JOIN workspace.healthcare_staffing.silver_monthly_patient_volume AS p
  ON s.month = p.month
  AND s.hospital_id = p.hospital_id
  AND s.department_id = p.department_id;

num_affected_rows,num_inserted_rows
